# Filter-bank стекинг

Под каждую частотную полосу свой римановский эксперт (перецентрирование + касательное пространство + LogReg).
Они смешиваются взвешенным геометрическим средним: $p(y\mid X) \propto \exp\big(\sum_b w_b \log p_b(y\mid x_b)\big)$, $w_b \ge 0$, $\sum_b w_b = 1$.
Веса учатся на предсказаниях «вне человека»: каждого train-участника предсказывают эксперты, обученные без него.
$w_b \ge 0$ не даёт перевернуть эксперта, поэтому смесь не ломается на знакомых людях.
Веса по 6 людям шумные, поэтому `shrink` подтягивает их к равным: 0 — как выучены, 1 — простое среднее. Стоит сравнить 0 / 0.5 / 1 по кросс-валидации.
Код: `bci/models/fbstack.py`.

In [ ]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from bci.data import load_split, concat
from bci.evaluation import score, evaluate, cross_validation_summary, compare_with_baselines
from bci.models import fbstack

train, val, test = (load_split(s, "../data") for s in ["train", "val", "test"])
dev = concat(train, val)  # для кросс-валидации; test только в самом конце

PARAMS = dict(bands=fbstack.BANDS, C=0.01, shrink=0.5, prior_weight=5)
fit = lambda tr: fbstack.fit(tr, **PARAMS)

## Какие полосы важны
Слева — каждый эксперт сам по себе на val (новые люди). Справа — вес $w_b$ каждой полосы в смеси.

In [ ]:
model = fit(train)
x_va, _ = model.features.transform(val)
log_p = np.log(np.stack([e.predict_proba(x_va[:, b]) for b, e in enumerate(model.experts)], axis=1))
names = [f"{lo}-{hi} Hz" for lo, hi in model.bands]

alone = {n: score(val, log_p[:, b].argmax(-1))["session_mean"] for b, n in enumerate(names)}
alone["ансамбль"] = score(val, np.einsum("nbk,b->nk", log_p, model.weights).argmax(-1))["session_mean"]

fig, (a, b) = plt.subplots(1, 2, figsize=(12, 3.5))
pd.Series(alone).plot.barh(ax=a, title="val session_mean"); a.axvline(1 / 3, ls="--", c="gray")
pd.Series(model.weights, index=names).plot.barh(ax=b, title="вес полосы $w_b$")
plt.tight_layout()

## Кросс-валидация на train + val
По ней сравниваем гипотезы: новые люди (каждый по очереди в тесте) и поздние сессии знакомых людей.

In [ ]:
cross_validation_summary(fit, dev)

## Официальный test
Обучение на train, одна оценка на test, рядом опубликованные бейзлайны.

In [ ]:
compare_with_baselines({"Filter-bank stacking": evaluate(fit(train), test)})